Here are 12 project ideas organized across six themes — you can filter by theme using the chips, and click any card to go deeper on a specific project.

A few notes on where to start depending on your team's maturity:

**Quick wins (2–4 weeks):** the white-space opportunity map and seasonal decomposition both rely mostly on data you already have and produce a visualization stakeholders immediately understand. Good for building credibility.

**High strategic value:** the rep visit impact detector and campaign attribution model are the two that directly answer the "is our marketing spend working" question. They're harder to execute correctly (causal inference, not just correlation) but have the clearest path to budget decisions.

**Long-running platform projects:** territory optimization and HCP segmentation are best treated as living systems — built once but refreshed quarterly as new sales data comes in.

On the technical stack, the 16-quarter history is particularly useful because it gives you enough data to separate true seasonality from trend in the time-series work, and enough behavioral history on HCPs to build stable segments. The ZIP-level granularity is valuable for the geo projects but will create sparse data problems in some ZIPs — hierarchical smoothing or borrowing strength from the county level will be important there.

Want me to go deep on any specific project — data requirements, model architecture, or how to frame the business case?

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Great choice — this is one of the more technically satisfying projects in the portfolio. Let me walk through each layer.

---

#### The core idea: hierarchical forecasting

The fundamental challenge is that you have sales data at multiple levels — national, state, county, ZIP — and you want forecasts that are *consistent* across levels (ZIP forecasts should sum up to county, county to state, state to national). That requires a specific modeling strategy called reconciliation, not just independent models at each level.

Here's the end-to-end data pipeline and model architecture:---

#### Data pipeline in detail

The key input tables you'll need to assemble:

| Table | Granularity | Key fields |
|---|---|---|
| Sales / Rx | ZIP × quarter | units, TRx, NRx, product |
| Rep activity | Rep × HCP × date | visit type, duration, outcome |
| Call log | HCP × date | call duration, outcome, channel |
| HCP registry | HCP ID | specialty, ZIP, decile, affiliation |
| Promo calendar | product × date | campaign type, spend, reach |
| Geography crosswalk | ZIP | county FIPS, state, region |

The geography crosswalk is often underestimated — ZIP codes change over time (USPS retires ~2–3% per year), and the ZIP → county mapping is many-to-many at boundaries. You'll want to use ZCTA (ZIP Code Tabulation Areas) from the Census Bureau rather than raw USPS ZIPs as your stable geographic unit.

---

#### Model layer — the three-zone problem

The hierarchy creates three distinct modeling zones, each with different data density:

The national and state levels have 16 clean quarters of data — enough for classical time series (Prophet, ARIMA-X, ETS). At this level you can explicitly model seasonality (flu season bumps, Q4 pullbacks, launch events), trend, and promotional regressors.

The county level is sparse for smaller markets. A panel approach — one LightGBM or XGBoost model across all counties with county-level embeddings — works much better than fitting 3,000 independent ARIMA models. You get cross-county learning and don't fail on sparse counties.

ZIP is the hardest. Many ZIPs will have zero sales in several quarters. You need intermittent demand methods (Croston's method, ADIDA) for those, and a flag/routing system that decides which ZIP gets a full time series model vs an intermittent model vs just propagated from its county-level forecast.

---

#### The reconciliation step (MinT)

This is the technical centerpiece. After running base models at each level independently, their forecasts almost certainly won't be coherent — the sum of your ZIP forecasts won't match your state forecast. MinT (Minimum Trace reconciliation) solves this by finding the smallest adjustment to all forecasts simultaneously that makes them sum correctly, weighted by each model's historical error covariance. The key insight is it doesn't just force bottom-up aggregation — it uses information from all levels to improve accuracy at all levels.

In Python, the `hierarchicalforecast` library (from Nixtla) handles this with a few lines of code once your base forecasts and the hierarchy S-matrix are set up.

---

#### Business deliverables

The outputs stakeholders will actually use:

**Quarterly forecast package** — a dashboard showing the next 2–4 quarters at each geo level, with 80% and 95% prediction intervals. State-level managers see their state; national sees rollup. Variance from plan is highlighted in red/amber.

**Geo opportunity heatmap** — which states/counties are forecasted to grow fastest vs plateau. This feeds directly into rep territory decisions and where to focus campaigns next quarter.

**Forecast accuracy scorecard** — rolling MAPE (mean absolute percentage error) at each geo level, tracked each quarter as actuals come in. This is your model's report card, and it's what builds or erodes credibility with the business.

**Launch scenario tool** — if you're launching a new product or entering a new geography, you can use the hierarchical structure to simulate a top-down scenario ("assume 3% national share in Q3") and distribute it down through the hierarchy using historical penetration patterns.

---

#### Pitfalls to plan for

The sparse ZIP problem will consume the most debugging time. A good rule of thumb: any ZIP with fewer than 6 non-zero quarters in your 16-quarter history should be flagged for the intermittent demand path.

Rep territory realignments mid-history are a silent killer — if territories were redrawn in, say, 2022, your ZIP-level rep coverage features have a discontinuity you must handle with a dummy variable or a data truncation decision.

COVID quarters (2020–2021) likely contain structural breaks in HCP visit patterns that don't represent the future. Consider weighting those quarters down or treating them as an outlier regime via a state-space model.

Want me to go deeper on any specific part — the MinT reconciliation math, the LightGBM panel model setup, or how to structure the forecast accuracy dashboard?

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

I'll give you clean, well-commented code for all three pieces. This assumes you're working in Python with pandas and the Nixtla ecosystem.

---

### National level — Prophet with regressorsThree files, each self-contained. Here's a quick orientation for each:

---

#### `national_model.py` — Prophet with regressors

The key design decisions made here: `seasonality_mode="multiplicative"` because pharma sales tend to grow proportionally (a 10% seasonal dip on a large base is a bigger absolute number than on a small base). `changepoint_prior_scale=0.05` is deliberately conservative — with only 16 quarters you don't want the model over-fitting structural breaks that may be noise. The COVID flag is added as a regressor rather than dropping those quarters, which preserves the data while letting the model learn to discount that regime.

---

#### `zip_model.py` — LightGBM panel + Croston's routing

The routing logic (`classify_zip`) is the most important structural decision in this file. The thresholds (10+ non-zero quarters → LightGBM, 4–9 → Croston's, below 4 → propagate from county) are a reasonable starting point but worth tuning based on your actual sparsity distribution. The LightGBM panel model shares parameters across all dense ZIPs — this is what gives you cross-ZIP learning and prevents the model from over-fitting tiny markets. The `min_child_samples=20` param is particularly important: it stops the model from memorizing any single ZIP's quirks.

---

#### `mint_reconciliation.py` — MinT shrinkage

The `mint_shrink` method is the right default for a large sparse hierarchy like yours. Plain `ols` or `wls_struct` can become numerically unstable when you have tens of thousands of ZIP nodes and many sparse series, because the covariance matrix estimation gets noisy. Shrinkage regularizes it toward a diagonal structure. The `check_coherence()` function at step 6 is worth keeping in production — it gives you a fast sanity check every time you run the pipeline that your ZIP-level outputs are actually summing to their county parents within tolerance.

The one thing you'll need to build out for production use is the "future feature rows" step in `zip_model.py` (noted in the comment at step 7) — constructing the feature matrix for the 4 future quarters requires knowing your planned rep visit counts and promo spend at ZIP level, or falling back to the rolling mean of recent quarters as a proxy.

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

The network is restricted in this sandbox so I can't install the packages here, but both files are complete and ready to run in your own environment. Let me walk through every step in detail.Install dependencies with `pip install statsmodels lightgbm scikit-learn pandas matplotlib pyarrow`.

---

## national_arima.py — step by step

**Step 1 — Synthetic data.** Builds 16 quarters of national TRx as the sum of four real-world components: a linear growth trend (baseline rising from 100k to 135k), a quarterly seasonality multiplier (Q4 gets a +6% flu-season lift, Q2 dips -4%), a COVID shock in Q2–Q3 2020 that knocks 15–18% off visits, and normalized promotional spend as an external regressor. Gaussian noise is added on top. This is the kind of multi-component signal you'd actually see in pharma.

**Step 2 — ADF stationarity test.** ARIMA's `d` parameter (differencing order) must be chosen based on whether the series has a unit root. The Augmented Dickey-Fuller test checks this formally. If p > 0.05 the series is non-stationary and we need `d=1` (first difference). The code prints the conclusion automatically so you don't have to interpret the statistic manually.

**Step 3 — Train/test split.** Last 4 quarters are held out. This is a temporal split — never split time series randomly, because doing so lets future values bleed into training lag features, giving artificially good-looking MAPE.

**Step 4 — SARIMAX model.** The order `(1,1,1)` means: one autoregressive lag (p=1), one round of differencing (d=1, removes the trend), one moving-average term (q=1, smooths one-period shocks). The seasonal order `(1,0,1,4)` adds a mild quarterly seasonal AR and MA with period s=4. `promo_spend` and `covid_flag` are included as external regressors (the X in SARIMAX). With only 16 quarters, the model is kept deliberately parsimonious — complex ARIMA orders overfit on short series.

**Step 5 — Forecast.** `get_forecast()` gives both point estimates and a full confidence interval. Passing `exog_test` lets the model use your known/planned promo spend and COVID flag values for the 4 held-out quarters — this is the key advantage of ARIMAX over plain ARIMA in promotional industries.

**Steps 6–7 — Plots and export.** Two diagnostic panels: forecast vs actuals (you want the orange line close to the dashed blue), and residuals (you want them randomly scattered around zero with no pattern). The residual plot is your model validity check — any visible trend or autocorrelation in residuals means the model missed something. Output is saved to a standard parquet schema for MinT.

---

## county_lgbm.py — step by step

**Step 1 — Synthetic panel data.** Generates 50 counties × 16 quarters = 800 rows. Each county gets a random baseline (5k–40k TRx, mimicking small rural vs large urban counties), its own growth rate, and its own noise level. Three external features are generated with realistic distributions: `rep_visits` (Poisson-distributed, correlated with market size), `hcp_density` (HCPs per 10k population), and `med_income`. The same seasonal and COVID shocks as the national model are applied so the hierarchy is internally consistent.

**Step 2 — Feature engineering.** This is the most important step. LightGBM is a tree model — trees cannot extrapolate a trend by themselves (a tree just looks up a bucket). You have to hand the model trend information explicitly through lag features (`trx_lag1`, `trx_lag2`, `trx_lag4`) and a `time_index` counter. The `trx_roll4_mean` rolling mean is especially useful: it gives the model a smooth recent baseline to anchor forecasts against for counties where one quarter was noisy. `quarter_num` handles seasonality. County and state are label-encoded integers — LightGBM treats these as categorical splits.

**Step 3 — Temporal train/test split.** Same logic as national: split by date (`2023-01-01`), never randomly. NaN rows from early lags are dropped — these are the first 4 quarters of each county's history where rolling features can't be computed yet.

**Step 4 — Panel model training.** One LightGBM model covers all 50 counties. This is the core advantage over fitting 50 individual models: the model learns cross-county patterns — for example that Q4 is always higher regardless of which county, or that high `hcp_density` counties respond more to rep visits. `min_child_samples=10` is critical: it prevents the model from memorizing individual quirky quarters in small counties by requiring at least 10 rows before a leaf can be created. `regression_l1` (MAE loss) is more robust than MSE when a few large-volume counties would otherwise dominate the gradient.

**Step 5 — Two-level evaluation.** Overall MAPE gives you the headline number. Per-county MAPE is more actionable — it tells you which specific markets the model struggles with. In production, counties in the worst-5 list deserve manual inspection: is there a rep territory change, a local formulary event, or a data quality issue?

**Step 6 — Feature importance.** Gain-based importance ranks features by how much they reduce prediction error at each split. You'd typically expect `trx_lag1` and `trx_roll4_mean` to dominate (recent history is the best predictor of near-term future), with `rep_visits` and `hcp_density` appearing lower but still meaningful — that's the signal that validates including those features at all.

**Steps 7–8 — Plots and export.** The left panel shows one sample county's full history with the 4-quarter holdout forecast overlaid, with a vertical line at the train/test boundary. The right panel is the feature importance bar chart — the top 3 are highlighted in darker blue. Output parquet feeds into MinT reconciliation with the same schema as the national file.

---

## How the three models connect

| Level | Model | Library | Key challenge |
|---|---|---|---|
| National | SARIMAX `(1,1,1)(1,0,1,4)` | `statsmodels` | Short series, needs differencing |
| County | LightGBM panel | `lightgbm` | Cross-county learning, lag features |
| ZIP | LightGBM + Croston's | `lightgbm` | Sparsity routing |
| All levels | MinT shrinkage | `hierarchicalforecast` | Coherence enforcement |

Each model exports the same parquet schema (`geo_id`, `ds`, `forecast`, `level`), which is exactly what `mint_reconciliation.py` from the previous session consumes.